In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================
from influxdb_client import InfluxDBClient
import pandas as pd
import folium
import numpy as np


In [6]:
# ============================================================
# 2. INFLUXDB CONFIG
# ============================================================
url= "https://us-east-1-1.aws.cloud2.influxdata.com"
token= "c-I6FEfaywxWzv05JWFI0EfJqKsaF5ECLR1o4rZBrMuTniUnXDtP49luBWUzG5QdgmJQDkatoWSCcX4FiLIRbQ=="
org=  "861c8c194327e7ec"
bucket=  "rsb"
  

In [9]:
LAT_COL = "lat_scaled"
LON_COL = "lon_scaled"

# ============================================================
# 3. FETCH DATA FROM INFLUXDB
# ============================================================
client = InfluxDBClient(url=url, token=token, org=org)
query = f'''
from(bucket: "{bucket}")
  |> range(start: 0)
  |> filter(fn: (r) => r._measurement == "{measurement}")
'''

tables = client.query_api().query(query)

records = []
for table in tables:
    for r in table.records:
        records.append({
            "time": r.get_time(),
            "field": r.get_field(),
            "value": r.get_value()
        })

df = pd.DataFrame(records)
if df.empty:
    raise SystemExit("❌ No data found in InfluxDB")

NameError: name 'measurement' is not defined

In [8]:
# ============================================================
# 4. PIVOT FIELDS → COLUMNS
# ============================================================
df = df.pivot(index="time", columns="field", values="value").reset_index()

# ============================================================
# 5. CLEAN GPS DATA
# ============================================================
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
df = df.dropna(subset=[LAT_COL, LON_COL]).sort_values("time").reset_index(drop=True)

# Auto-scale if needed
if df[LAT_COL].abs().max() > 90:
    df[LAT_COL] /= 1e7
if df[LON_COL].abs().max() > 180:
    df[LON_COL] /= 1e7

# ============================================================
# 6. PICK 10 POINTS
# ============================================================
n_points = 50
if len(df) < n_points:
    points = df[[LAT_COL, LON_COL]].values.tolist()
else:
    indices = np.linspace(0, len(df)-1, n_points, dtype=int)
    points = df.iloc[indices][[LAT_COL, LON_COL]].values.tolist()

# ============================================================
# 7. CREATE FOLIUM MAP
# ============================================================
m = folium.Map(location=[points[0][0], points[0][1]], zoom_start=14)

# Route line
folium.PolyLine(points, color="red", weight=4, opacity=0.8).add_to(m)

# Add markers
for i, (lat, lon) in enumerate(points):
    if i == 0:
        label = "Start"
    elif i == len(points)-1:
        label = "End"
    else:
        label = f"Point {i+1}"
    folium.Marker([lat, lon], popup=label).add_to(m)

# Save map
m.save("gps_route_map.html")
print("✅ Saved gps_route_map.html — open in browser to view the route")

NameError: name 'df' is not defined

In [6]:
# ============================================================
# 7. CREATE FOLIUM MAP WITH TIMELINE STYLE
# ============================================================
from branca.colormap import linear

# Create map centered at first point
m = folium.Map(location=[points[0][0], points[0][1]], zoom_start=14)

# Generate a color gradient based on time
times = pd.to_datetime(df["time"])
min_time, max_time = times.min(), times.max()
colormap = linear.YlOrRd_09.scale(min_time.timestamp(), max_time.timestamp())

# Add points with timestamp popup and color
for i, row in df.iterrows():
    lat = row[LAT_COL]
    lon = row[LON_COL]
    timestamp = row["time"]
    color = colormap(pd.to_datetime(timestamp).timestamp())
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=str(timestamp)
    ).add_to(m)

# Connect points with a polyline
folium.PolyLine(df[[LAT_COL, LON_COL]].values.tolist(), color="blue", weight=2, opacity=0.5).add_to(m)

# Add color legend
colormap.caption = 'Time progression'
colormap.add_to(m)

# Save map
m.save("gps_route_timeline_map.html")
print("✅ Saved gps_route_timeline_map.html — open in browser to view the route")


✅ Saved gps_route_timeline_map.html — open in browser to view the route


In [7]:
# ============================================================
# Animated popup map
# ============================================================
m = folium.Map(location=[points[0][0], points[0][1]], zoom_start=14)

# Prepare JavaScript for animated markers
js = ""
for i, row in df.iterrows():
    lat = row[LAT_COL]
    lon = row[LON_COL]
    timestamp = row["time"]
    js += f"""
    setTimeout(function() {{
        var marker = L.circleMarker([{lat}, {lon}], {{
            radius: 6,
            color: 'red',
            fillColor: 'red',
            fillOpacity: 0.8
        }}).addTo({{map}});
        marker.bindPopup("Time: {timestamp}").openPopup();
    }}, {i*1000});  // delay in ms
    """

# Add JS to map
from folium import MacroElement
from jinja2 import Template

class AnimateMarkers(MacroElement):
    def __init__(self, js_code):
        super().__init__()
        self._template = Template(f"""
        <script>
        function addMarkers(map) {{
            {js_code.replace('{{map}}', 'map')}
        }}
        addMarkers(map);
        </script>
        """)

m.get_root().add_child(AnimateMarkers(js))

# Save map
m.save("gps_route_animated.html")
print("✅ Saved gps_route_animated.html — open in browser to watch the animation")


✅ Saved gps_route_animated.html — open in browser to watch the animation


In [8]:
from folium.plugins import TimestampedGeoJson
import json

# Prepare features for TimestampedGeoJson
features = []
for i, row in df.iterrows():
    lat = row[LAT_COL]
    lon = row[LON_COL]
    timestamp = pd.to_datetime(row["time"]).isoformat()
    
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {"time": timestamp, "popup": f"Time: {timestamp}", "icon": "circle", "iconstyle": {"fillColor": "red", "fillOpacity": 0.8, "stroke": "true", "radius": 6}}
    })

# Create map
m = folium.Map(location=[df[LAT_COL].iloc[0], df[LON_COL].iloc[0]], zoom_start=14)

# Add animated time series
TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="PT1S",            # 1-second interval per point
    add_last_point=True,
    auto_play=True,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm:ss",
    time_slider_drag_update=True
).add_to(m)

# Save map
m.save("gps_route_animated.html")
print("✅ Saved gps_route_animated.html — open in browser to watch the animation")



✅ Saved gps_route_animated.html — open in browser to watch the animation
